# Inspect PartEdit-Bench For Part-Level TDM Localization

This notebook checks whether PartEdit-Bench can support Harry Yang's requested pilot study: 10-15 cases balanced by target part size, with ground-truth masks for evaluating Follow-Your-Shape TDM localization.

## Goal

Before renting a GPU or running Follow-Your-Shape, confirm the dataset fields, image/mask formats, prompt structure, and mask-area distribution. The notebook should produce a small candidate table, not download or commit the full dataset into the repository.

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import get_dataset_config_names, load_dataset
from PIL import Image

REPO_ROOT = Path.cwd()
DATASET_ID = "Aleksandar/PartEdit-Bench"
OUTPUT_DIR = REPO_ROOT / "core" / "data" / "partedit_subset"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

## 1. Inspect Dataset Configs And Splits

In [ ]:
configs = get_dataset_config_names(DATASET_ID)
configs

In [ ]:
dataset_kwargs = {}
if configs:
    dataset_kwargs["name"] = configs[0]

dataset = load_dataset(DATASET_ID, **dataset_kwargs)
dataset

## 2. Inspect Fields

In [ ]:
split_name = "train" if "train" in dataset else list(dataset.keys())[0]
split = dataset[split_name]

print("split:", split_name)
print("num rows:", len(split))
print("features:")
split.features

In [ ]:
sample = split[0]
{key: type(value).__name__ for key, value in sample.items()}

In [ ]:
preview_rows = []
for idx in range(min(5, len(split))):
    row = split[idx]
    preview_rows.append({
        "index": idx,
        **{key: row[key] for key in row if isinstance(row[key], (str, int, float, bool, type(None)))}
    })

pd.DataFrame(preview_rows)

## 3. Identify Image And Mask Fields

Adjust these field names after inspecting the dataset features above. Keep the choices explicit so later scripts can reuse them.

In [ ]:
candidate_image_fields = [key for key, value in sample.items() if isinstance(value, Image.Image)]
candidate_array_fields = [key for key, value in sample.items() if isinstance(value, (list, np.ndarray))]
candidate_text_fields = [key for key, value in sample.items() if isinstance(value, str)]

print("PIL image fields:", candidate_image_fields)
print("array-like fields:", candidate_array_fields)
print("text fields:", candidate_text_fields)

In [ ]:
IMAGE_FIELD = candidate_image_fields[0] if candidate_image_fields else "image"
MASK_FIELD = "mask"
SOURCE_PROMPT_FIELD = "source_prompt"
TARGET_PROMPT_FIELD = "target_prompt"
PART_FIELD = "part"

required_guess = [IMAGE_FIELD, MASK_FIELD, SOURCE_PROMPT_FIELD, TARGET_PROMPT_FIELD]
missing_guess = [field for field in required_guess if field not in sample]
print("field guesses:", required_guess)
print("missing guessed fields:", missing_guess)

## 4. Visual Check One Example

In [ ]:
def as_mask_array(value):
    if isinstance(value, Image.Image):
        arr = np.asarray(value.convert("L"))
    else:
        arr = np.asarray(value)
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr > 0


def mask_area_ratio(mask_value):
    mask = as_mask_array(mask_value)
    return float(mask.mean())


if IMAGE_FIELD not in sample or MASK_FIELD not in sample:
    raise KeyError(f"Update IMAGE_FIELD/MASK_FIELD. Available fields: {list(sample.keys())}")

image = sample[IMAGE_FIELD]
mask = as_mask_array(sample[MASK_FIELD])

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image)
axes[0].set_title("source image")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title(f"target mask ratio={mask.mean():.3f}")
axes[2].imshow(image)
axes[2].imshow(mask, alpha=0.35, cmap="Reds")
axes[2].set_title("mask overlay")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## 5. Compute Mask-Area Distribution

In [ ]:
records = []
for idx in range(len(split)):
    row = split[idx]
    if MASK_FIELD not in row:
        raise KeyError(f"MASK_FIELD={MASK_FIELD!r} not found. Available fields: {list(row.keys())}")
    record = {
        "dataset_index": idx,
        "mask_area_ratio": mask_area_ratio(row[MASK_FIELD]),
    }
    for field in [SOURCE_PROMPT_FIELD, TARGET_PROMPT_FIELD, PART_FIELD]:
        if field in row and isinstance(row[field], (str, int, float, bool, type(None))):
            record[field] = row[field]
    records.append(record)

mask_table = pd.DataFrame(records).sort_values("mask_area_ratio")
mask_table.describe(include="all")

In [ ]:
ax = mask_table["mask_area_ratio"].hist(bins=30, figsize=(8, 4))
ax.set_title("PartEdit-Bench target mask area ratios")
ax.set_xlabel("mask area / image area")
ax.set_ylabel("case count")

## 6. Build A Candidate Balanced Subset

This produces a first-pass candidate table. Review examples visually before turning it into the final `cases.json` manifest.

In [ ]:
def size_bucket(mask_ratio):
    if mask_ratio < 0.05:
        return "small"
    if mask_ratio < 0.15:
        return "medium"
    return "large"


mask_table["part_size"] = mask_table["mask_area_ratio"].map(size_bucket)
candidate_subset = (
    mask_table.groupby("part_size", group_keys=False)
    .apply(lambda frame: frame.sort_values("mask_area_ratio").head(5))
    .reset_index(drop=True)
)

candidate_subset

In [ ]:
preview_path = OUTPUT_DIR / "cases_preview.csv"
candidate_subset.to_csv(preview_path, index=False)
preview_path

## Next Checks

- Confirm the guessed field names match the actual dataset schema.
- Visually inspect candidate source images and masks.
- Replace the preview CSV with a reviewed JSON manifest for the 10-15 pilot cases.
- Do not commit downloaded images or masks.